# Last-token probes: ternary training and ternary testing

Compare **t=1024, 262144, 1048576** from `ternary_checkpoints_depth_3`.
At each checkpoint, fit a separate linear probe on the last input position at the embedding and after layers 1–3, predicting the held-out final token.

Both probe datasets contain **only fully ternary trees** (27 leaves, 26 input tokens). Use 32,768 examples per split, independently sampled with seeds 42 and 43; reuse these fixed splits across checkpoints. The transformer has 3 layers, width 768, and 16 heads.

Run all cells from the repository root. Results and figures use dedicated ternary/ternary filenames. The plot retains the previous theoretical accuracy plateaus, skipping the first (0.417308) plateau. Cached results are reused when their metadata matches; set `RECOMPUTE = True` to refit.


## 1. Setup and fully ternary probe datasets


In [ ]:
import gc, sys, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'models').exists(): raise RuntimeError('Run from the repository root.')
sys.path[:0] = [str(WORKSPACE), str(WORKSPACE / 'datasets' / 'rhm')]
import models
from mixed_random_hierarchy_model import (
    sample_padded_rules,
    sample_data_from_labels_varying_tree_tensorized,
)

V, L, BIN = 8, 3, 2
M_2, M_3, D_MAX, PAD_TOKEN = BIN, BIN * V, 3 ** L, V
T = D_MAX - 1
MODEL_DEPTH = 3
STEPS = [1024, 262144, 1048576]
SEED_RULES, TRAIN_SEED, TEST_SEED = 12345678, 42, 43
N_TRAIN, N_TEST = 32768, 32768
NUM_HEADS, BATCH_SIZE, PROBE_C = 16, 256, 0.1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
CHECKPOINT_DIR = WORKSPACE / 'ternary_checkpoints_depth_3'
CHECKPOINTS = {step: CHECKPOINT_DIR / f'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth3_demb_768_nheads_16_arr0_t{step}.pt' for step in STEPS}
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_ternary_train_ternary_test_last_token_depth3.pkl' for step in STEPS}
RECOMPUTE = False  # Reuse results only for this ternary/ternary experiment.
# Cached stages need no checkpoint file.
missing = [CHECKPOINTS[step] for step in STEPS
           if (RECOMPUTE or not RESULT_PATHS[step].is_file())
           and not CHECKPOINTS[step].is_file()]
if missing: raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(map(str, missing)))

rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)
ternary_probs = {l: torch.cat([torch.zeros(M_2), torch.full((M_3,), 1/M_3)]) for l in range(L)}

def generate_sentences(n, probabilities, seed):
    state = torch.random.get_rng_state(); torch.manual_seed(seed)
    labels = torch.randint(0, V, (n,))
    sentences, _ = sample_data_from_labels_varying_tree_tensorized(labels, rules, probabilities, V, D_MAX)
    torch.random.set_rng_state(state)
    return sentences

def make_last_token_task(sentences):
    # Mirrors init.MySentenceDataLoader exactly.
    n = len(sentences); lengths = (sentences != PAD_TOKEN).sum(1)
    targets = sentences[torch.arange(n), lengths - 1]
    input_lengths = lengths - 1
    columns = torch.arange(T).unsqueeze(0).expand(n, T)
    source_columns = columns - T + input_lengths.unsqueeze(1)
    gathered = torch.gather(sentences[:, :T], 1, source_columns.clamp(min=0))
    inputs = torch.where(source_columns >= 0, gathered, torch.full_like(gathered, PAD_TOKEN))
    return inputs, targets, lengths

train_sentences = generate_sentences(N_TRAIN, ternary_probs, TRAIN_SEED)
test_sentences = generate_sentences(N_TEST, ternary_probs, TEST_SEED)
X_train_tokens, y_train, train_lengths = make_last_token_task(train_sentences)
X_test_tokens, y_test, test_lengths = make_last_token_task(test_sentences)
assert torch.all(test_lengths == D_MAX)
assert torch.all(train_lengths == D_MAX)
assert not torch.any(X_train_tokens == PAD_TOKEN)
assert not torch.any(X_test_tokens == PAD_TOKEN)
assert torch.all(y_train < V) and torch.all(y_test < V)
print(f'device={DEVICE}; ternary train={N_TRAIN:,}; ternary test={N_TEST:,}')
print(f'train lengths={train_lengths.min().item()}..{train_lengths.max().item()}; ternary lengths={test_lengths.unique().tolist()}')
CACHE_METADATA = {
    'experiment': 'ternary_train_ternary_test_last_token_depth3_v1',
    'n_train_ternary': N_TRAIN, 'n_test_ternary': N_TEST,
    'train_seed': TRAIN_SEED, 'test_seed': TEST_SEED,
    'rules_seed': SEED_RULES, 'probe_C': PROBE_C,
    'probe_position': 'last input position',
}

def cache_matches(saved, step):
    return (all(saved.get(key) == value for key, value in CACHE_METADATA.items())
            and saved.get('checkpoint_step') == step
            and saved.get('checkpoint_file') == CHECKPOINTS[step].name)


## 2. Model loading, last-position activations, and linear probes


In [ ]:
def infer_arch(state_dict):
    width = state_dict['token_embedding_table.weight'].shape[1]
    n_layers = max(int(k.split('.')[1]) for k in state_dict if k.startswith('blocks.')) + 1
    return dict(vocab_size=state_dict['token_embedding_table.weight'].shape[0],
                block_size=state_dict['blocks.0.attn.tril'].shape[0],
                embedding_dim=width, num_heads=NUM_HEADS,
                ffwd_size=state_dict['blocks.0.ffwd.net.0.weight'].shape[0] // width,
                num_layers=n_layers, dropout=0.0,
                rope='position_embedding_table.weight' not in state_dict, share_emb=False)

def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    # Compiled checkpoints prefix parameter names with _orig_mod.
    state_dict = {key.removeprefix('_orig_mod.'): value
                  for key, value in saved['model'].items()}
    arch = infer_arch(state_dict); assert arch['block_size'] == T
    if (arch['num_layers'], arch['embedding_dim']) != (MODEL_DEPTH, 768):
        raise ValueError(f'Expected 3 layers and embedding width 768, got {arch}')
    model = models.CLM(**arch); model.load_state_dict(state_dict)
    return model.to(DEVICE).eval(), arch, int(saved.get('step', path.stem.rsplit('_t', 1)[-1]))

@torch.no_grad()
def extract_last_position_activations(model, inputs):
    # Shape: [embedding + blocks, examples, hidden width]. No other token positions are retained.
    by_depth = [[] for _ in range(model.num_layers + 1)]
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = inputs[start:start + BATCH_SIZE].to(DEVICE); captured = {}
        hooks = [model.blocks[0].register_forward_pre_hook(
            lambda module, args: captured.update({0: args[0][:, -1].detach().cpu()}))]
        for depth, block in enumerate(model.blocks, 1):
            hooks.append(block.register_forward_hook(
                lambda module, args, output, depth=depth: captured.update({depth: output[:, -1].detach().cpu()})))
        model(batch)
        for hook in hooks: hook.remove()
        for depth in range(model.num_layers + 1): by_depth[depth].append(captured[depth].numpy())
    return np.stack([np.concatenate(chunks) for chunks in by_depth])

def probe_across_depth(train_acts, test_acts):
    results = {'train_accuracy': [], 'test_accuracy': [],
               'train_cross_entropy_normalized': [], 'test_cross_entropy_normalized': []}
    train_labels, test_labels = y_train.numpy(), y_test.numpy()
    for depth, (X_train, X_test) in enumerate(zip(train_acts, test_acts)):
        probe = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(
            C=PROBE_C, solver='lbfgs', max_iter=5000, random_state=0))])
        probe.fit(X_train, train_labels)
        results['train_accuracy'].append(probe.score(X_train, train_labels))
        results['test_accuracy'].append(probe.score(X_test, test_labels))
        train_ce = log_loss(train_labels, probe.predict_proba(X_train), labels=np.arange(V)) / np.log(V)
        test_ce = log_loss(test_labels, probe.predict_proba(X_test), labels=np.arange(V)) / np.log(V)
        results['train_cross_entropy_normalized'].append(train_ce)
        results['test_cross_entropy_normalized'].append(test_ce)
        print(f"  depth {depth}: acc train={results['train_accuracy'][-1]:.4f}, test={results['test_accuracy'][-1]:.4f}; "
              f"CE/log(V) train={train_ce:.4f}, test={test_ce:.4f}")
    counts = np.bincount(train_labels, minlength=V)
    results.update(chance_uniform=1/V, train_majority_baseline=counts.max()/counts.sum())
    return results

## 3. Run the three checkpoints

Save accuracy and normalized cross-entropy for each probe depth.


In [ ]:
token_probe_results = {}
architecture = None
for requested_step in STEPS:
    print(f'\n=== checkpoint t={requested_step} ===')
    if RESULT_PATHS[requested_step].exists() and not RECOMPUTE:
        with RESULT_PATHS[requested_step].open('rb') as handle: cached = pickle.load(handle)
        cached_result = cached.get('token_probe', cached)
        if (cache_matches(cached, requested_step)
                and 'test_cross_entropy_normalized' in cached_result
                and all(len(cached_result.get(metric, [])) == MODEL_DEPTH + 1
                        for metric in ('test_accuracy', 'test_cross_entropy_normalized'))):
            token_probe_results[requested_step] = np.asarray(cached_result['test_accuracy'], dtype=float)
            print('reusing:', RESULT_PATHS[requested_step])
            continue
        print('cached metadata or metrics differ; refitting this checkpoint')
    model, arch, loaded_step = load_model(CHECKPOINTS[requested_step])
    if loaded_step != requested_step: raise ValueError(f'Expected step {requested_step}, checkpoint reports {loaded_step}')
    if architecture is None: architecture = arch
    elif (arch['num_layers'], arch['embedding_dim']) != (architecture['num_layers'], architecture['embedding_dim']):
        raise ValueError('Checkpoint architectures differ.')
    train_acts = extract_last_position_activations(model, X_train_tokens)
    test_acts = extract_last_position_activations(model, X_test_tokens)
    result = probe_across_depth(train_acts, test_acts)
    token_probe_results[requested_step] = np.asarray(result['test_accuracy'], dtype=float)
    payload = {**CACHE_METADATA, 'token_probe': result, 'checkpoint_step': requested_step,
               'checkpoint_file': CHECKPOINTS[requested_step].name, 'architecture': arch}
    with RESULT_PATHS[requested_step].open('wb') as handle: pickle.dump(payload, handle)
    print('saved:', RESULT_PATHS[requested_step])
    del model, train_acts, test_acts; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    elif DEVICE == 'mps': torch.mps.empty_cache()

## 4. Last-token probe accuracy across layers

This cell can run independently once results are saved. Export PDF and PNG with a curve for each checkpoint and the same Inside/Outside divider as before.


In [ ]:
# Can also run this cell alone after the probe results have been saved.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
STEPS = [1024, 262144, 1048576]
MODEL_DEPTH, V = 3, 8
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_ternary_train_ternary_test_last_token_depth3.pkl'
                for step in STEPS}
curves = {}
for step, path in RESULT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Run the probe cells first; missing {path}')
    with path.open('rb') as handle:
        saved = pickle.load(handle)
    if (saved.get('experiment') != 'ternary_train_ternary_test_last_token_depth3_v1'
            or saved.get('checkpoint_step') != step):
        raise ValueError(f'Unexpected experiment or checkpoint in {path}')
    curve = np.asarray(saved['token_probe']['test_accuracy'], dtype=float)
    if curve.shape != (MODEL_DEPTH + 1,) or not np.all(np.isfinite(curve)) or np.any((curve < 0) | (curve > 1)):
        raise ValueError(f'Invalid accuracy curve in {path}')
    curves[step] = curve

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
depth = np.arange(MODEL_DEPTH + 1)
fig, ax = plt.subplots(figsize=(3.5, 3.45))
for stage, step in enumerate(STEPS, 1):
    ax.plot(depth, curves[step], marker='o', lw=1.8, ms=3.8,
            label=f'Stage {stage} (t={step:,})')
# Retain the old theoretical plateaus, excluding the first (VT trigram).
accuracy_plateaus = [
    (0.507880, r'$A_{\mathrm{trig}}^{\mathrm{FT}}$'),
    (0.737, r'$A_{2}$'),
    (0.851, r'$A_{3}$'),
]
for value, label in accuracy_plateaus:
    ax.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax.text(MODEL_DEPTH + .13, value + .008, label,
            color='black', fontsize=8.5, va='bottom')
ax.axhline(1 / V, color='grey', ls=':', lw=1, label='Chance')
divider = MODEL_DEPTH / 2
#ax.axvline(divider, color='black', ls='--', lw=.9, alpha=.5)
# for x, label, align in [(divider - .16, 'Inside', 'right'), (divider + .16, 'Outside', 'left')]:
#     ax.text(x, .975, label, transform=ax.get_xaxis_transform(), ha=align, va='top', fontsize=8)
ax.set(xticks=depth, xticklabels=['emb', 'L1', 'L2', 'L3'],
       xlim=(-.2, MODEL_DEPTH + 1.55), ylim=(0, 1.02),
       xlabel='Transformer depth', ylabel='Last-token probe accuracy',
       title=r'Fixed tree training $s=3$')
ax.grid(alpha=.16, lw=.55)
fig.legend(*ax.get_legend_handles_labels(), loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(.5, 1.01), columnspacing=1.0)
fig.tight_layout(rect=(0, 0, 1, .83))
for extension in ('pdf', 'png'):
    path = WORKSPACE / f'token_probe_accuracy_ternary_train_ternary_test_across_depth_depth3.{extension}'
    fig.savefig(path, dpi=300)
    print('saved:', path)
plt.show()


## 5. Last-token probe cross-entropy across layers

Uses the cached test cross-entropy normalized by `log(V)` (`V=8`). Lower is better; uniform predictions give 1. Multiply by `log(8)` to recover cross-entropy in nats. Dashed lines show the supplied theoretical cross-entropy plateaus: $\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}=0.452838$, $\mathcal{L}_2=0.205742$, and $\mathcal{L}_3=0.115976$. This cell runs independently and exports PDF and PNG.

The y-axis uses a logarithmic scale.


In [ ]:
# Run independently using the saved probe results; no model loading or refitting.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
STEPS = [1024, 262144, 1048576]
MODEL_DEPTH, V = 3, 8
curves_ce = {}
for step in STEPS:
    path = WORKSPACE / f'probe_results_step_{step}_ternary_train_ternary_test_last_token_depth3.pkl'
    if not path.is_file():
        raise FileNotFoundError(f'Run the probe cells first; missing {path}')
    with path.open('rb') as handle:
        saved = pickle.load(handle)
    if (saved.get('experiment') != 'ternary_train_ternary_test_last_token_depth3_v1'
            or saved.get('checkpoint_step') != step):
        raise ValueError(f'Unexpected experiment or checkpoint in {path}')
    curve = np.asarray(saved['token_probe']['test_cross_entropy_normalized'], dtype=float)
    if curve.shape != (MODEL_DEPTH + 1,) or not np.all(np.isfinite(curve)) or np.any(curve <= 0):
        raise ValueError(f'Invalid cross-entropy curve in {path}')
    curves_ce[step] = curve

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
depth = np.arange(MODEL_DEPTH + 1)
fig, ax = plt.subplots(figsize=(3.5, 3.45))
for stage, step in enumerate(STEPS, 1):
    ax.plot(depth, curves_ce[step], marker='o', lw=1.8, ms=3.8,
            label=f'Stage {stage}')
plateaus = {
    r"$\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}$": 0.452838,
    r"$\mathcal{L}_2$": 0.205742,
    r"$\mathcal{L}_3$": 0.115976,
}
for label, value in plateaus.items():
    ax.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax.text(MODEL_DEPTH + .13, value * 1.025, label,
            color='black', fontsize=8.5, va='bottom')
ax.axhline(1, color='grey', ls=':', lw=1, label='Uniform prediction')
ax.set(xticks=depth, xticklabels=[r'emb ($k=0$)', r'$k=1$', r'$k=2$', r'$k=3$'],
       xlim=(-.2, MODEL_DEPTH + .9),
       ylim=(min(min(curve.min() for curve in curves_ce.values()), min(plateaus.values())) * .8,
             max(1, max(curve.max() for curve in curves_ce.values())) * 1.1),
       yscale='log',
       xlabel=r'Transformer layer $k$', ylabel=r'Probe loss $\mathcal{L}_{x_d}^{(k,t)}$',
       title='Fixed-tree training')
ax.grid(alpha=.16, lw=.55)
fig.legend(*ax.get_legend_handles_labels(), loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(.5, 1.01), columnspacing=1.0)
fig.tight_layout(rect=(0, 0, 1, .83))
for extension in ('pdf', 'png'):
    path = WORKSPACE / f'token_probe_cross_entropy_ternary_train_ternary_test_across_depth_depth3.{extension}'
    fig.savefig(path, dpi=300)
    print('saved:', path)
plt.show()
